# Tugas Besar II4013 Data Analytics
## **OSEMN** 
Proyek ini disusun dengan OSEMN Framework: *Obtain, Scrub, Explore, Model, dan iNterpret*. Framework ini menjadi dasar alur kerja teknis, struktur laporan, struktur notebook script, dan urutan presentasi.  

**Anggota Kelompok:**

| Nama | NIM |
|:----:|:---:|
| Bryan P. Hutagalung | 18222130 |
| Naila Selvira Budiana | 18223018 |
| Florecita Natawirya | 18223040 |
| Fhatika Adhalisman Ryanjani | 18223068 |
| Muhammad Refino Ramadhan | 18223070 |

# **1. Obtain**

## *Data Engineering* / 18223040

#### 1. Scope Obtain

Bagian ini menjadi bukti proses **Data Engineering** pada tahap **Obtain**. Fokusnya adalah mengambil data kemiskinan dan kerentanan sosial Jawa Barat langsung dari API Open Data Jabar, menyimpan hasil mentah secara reproducible, dan menyediakan orchestration scaffold agar pipeline bisa dijalankan ulang saat sumber data diperbarui.

Notebook ini tidak mengubah script ekstraksi milik data engineer. Sel di bawah hanya membaca atau memanggil `scripts/extract_api.py` dan `scripts/orchestrate_prefect.py` sebagai proof proses.


#### Checklist Data Engineering

| Kriteria Obtain / Data Engineering | Implementasi di repository | Bukti di notebook |
|---|---|---|
| Mengakses data dari sumber eksternal | Open Data Jabar via dokumen OpenAPI dan endpoint record aktual | Katalog dataset dan endpoint API |
| Automasi extraction | `scripts/extract_api.py` mengambil semua record dengan pagination `limit` dan `skip` | Sel opsi run extraction |
| Penyimpanan raw data | JSON dan CSV mentah disimpan di `data/raw/`, termasuk versi timestamp dan file latest | Audit file raw |
| Logging dan manifest | `logs/ingest.log` dan `data/manifest/ingest_manifest.csv` mencatat status, source URL, checksum | Ringkasan manifest |
| Reproducible pipeline | Script dapat dijalankan ulang; checksum mencegah duplikasi ingest yang sama | Status manifest dan idempotensi |
| Orkestrasi | Scaffold Prefect memanggil script extraction | Validasi isi `orchestrate_prefect.py` |


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "extract_api.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts import extract_api as extract_module

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MANIFEST_PATH = PROJECT_ROOT / "data" / "manifest" / "ingest_manifest.csv"
EXTRACT_SCRIPT = PROJECT_ROOT / "scripts" / "extract_api.py"
ORCHESTRATE_SCRIPT = PROJECT_ROOT / "scripts" / "orchestrate_prefect.py"

print(f"Project root     : {PROJECT_ROOT}")
print(f"Extract script   : {EXTRACT_SCRIPT.relative_to(PROJECT_ROOT)}")
print(f"Orchestrator     : {ORCHESTRATE_SCRIPT.relative_to(PROJECT_ROOT)}")
print(f"Raw data folder  : {RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Manifest         : {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")


Project root     : /Users/refinormdn/data-analytics-tubes
Extract script   : scripts/extract_api.py
Orchestrator     : scripts/orchestrate_prefect.py
Raw data folder  : data/raw
Manifest         : data/manifest/ingest_manifest.csv


#### 2. Katalog Dataset Obtain

Dataset yang dipakai berasal dari Open Data Jabar dan dipilih agar sesuai dengan fokus analytics: tren kemiskinan, kerentanan sosial, dan prioritas intervensi wilayah di Jawa Barat. Lima dataset ini nantinya menjadi input untuk tahap Scrub/preprocessing.


In [2]:
dataset_notes = {
    "raw_garis_kemiskinan": {
        "kelompok": "Kemiskinan",
        "indikator": "Garis kemiskinan",
        "alasan": "Mengukur batas minimum pengeluaran penduduk miskin per wilayah.",
    },
    "raw_persentase_miskin": {
        "kelompok": "Kemiskinan",
        "indikator": "Persentase penduduk miskin",
        "alasan": "Menjadi indikator utama tren kemiskinan kabupaten/kota.",
    },
    "raw_keparahan_kemiskinan": {
        "kelompok": "Kemiskinan",
        "indikator": "Indeks keparahan kemiskinan",
        "alasan": "Menunjukkan kedalaman masalah pada kelompok miskin.",
    },
    "raw_ipm_sp2010": {
        "kelompok": "Kerentanan sosial",
        "indikator": "Indeks Pembangunan Manusia SP2010",
        "alasan": "Mewakili kualitas pembangunan manusia dan kapasitas sosial wilayah.",
    },
    "raw_pengangguran_terbuka": {
        "kelompok": "Kerentanan sosial",
        "indikator": "Tingkat pengangguran terbuka",
        "alasan": "Mewakili tekanan sosial-ekonomi yang dapat memperbesar kerentanan.",
    },
}

def latest_raw_payload(logical_name):
    json_path = RAW_DIR / f"{logical_name}.json"
    if not json_path.exists():
        return {}
    return json.loads(json_path.read_text(encoding="utf-8"))

def summarize_latest_csv(logical_name):
    csv_path = RAW_DIR / f"{logical_name}.csv"
    if not csv_path.exists():
        return {"raw_file": None, "rows": 0, "columns": 0, "tahun_min": None, "tahun_max": None}
    df = pd.read_csv(csv_path)
    tahun = pd.to_numeric(df.get("tahun"), errors="coerce") if "tahun" in df.columns else pd.Series(dtype="float64")
    return {
        "raw_file": csv_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "tahun_min": int(tahun.min()) if tahun.notna().any() else None,
        "tahun_max": int(tahun.max()) if tahun.notna().any() else None,
    }

catalog_rows = []
for logical_name, doc_url in extract_module.api_docs.items():
    payload = latest_raw_payload(logical_name)
    row = {
        "logical_name": logical_name,
        "kelompok": dataset_notes[logical_name]["kelompok"],
        "indikator": dataset_notes[logical_name]["indikator"],
        "alasan_pemilihan": dataset_notes[logical_name]["alasan"],
        "source_doc_url": doc_url,
        "source_data_url": payload.get("source_data_url"),
    }
    row.update(summarize_latest_csv(logical_name))
    catalog_rows.append(row)

catalog_df = pd.DataFrame(catalog_rows)
catalog_df

,logical_name,kelompok,indikator,alasan_pemilihan,source_doc_url,source_data_url,raw_file,rows,columns,tahun_min,tahun_max
0,raw_garis_kemiskinan,Kemiskinan,Garis kemiskinan,Mengukur batas minimum pengeluaran penduduk mi...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_garis_kemiskinan.csv,594,8,2004,2025
1,raw_persentase_miskin,Kemiskinan,Persentase penduduk miskin,Menjadi indikator utama tren kemiskinan kabupa...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_persentase_miskin.csv,405,8,2010,2024
2,raw_keparahan_kemiskinan,Kemiskinan,Indeks keparahan kemiskinan,Menunjukkan kedalaman masalah pada kelompok mi...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_keparahan_kemiskinan.csv,567,8,2004,2024
3,raw_ipm_sp2010,Kerentanan sosial,Indeks Pembangunan Manusia SP2010,Mewakili kualitas pembangunan manusia dan kapa...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_ipm_sp2010.csv,405,8,2010,2024
4,raw_pengangguran_terbuka,Kerentanan sosial,Tingkat pengangguran terbuka,Mewakili tekanan sosial-ekonomi yang dapat mem...,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,raw_pengangguran_terbuka.csv,424,8,2007,2024


#### 3. Cara Akses Data API

Script extraction memakai pendekatan API-first. URL yang didefinisikan di script adalah dokumen OpenAPI; dari dokumen tersebut script mencari endpoint data aktual, lalu mengambil semua record dengan pagination. Output JSON raw juga menyimpan `source_doc_url`, `source_data_url`, metadata jumlah baris, limit, dan jumlah halaman yang berhasil diambil.


In [3]:
endpoint_rows = []
for logical_name in extract_module.api_docs:
    payload = latest_raw_payload(logical_name)
    metadata = payload.get("metadata", {})
    endpoint_rows.append({
        "logical_name": logical_name,
        "source_doc_url": payload.get("source_doc_url", extract_module.api_docs[logical_name]),
        "source_data_url": payload.get("source_data_url"),
        "rows_from_json": metadata.get("rows"),
        "pages": metadata.get("pages"),
        "limit": metadata.get("limit"),
        "ingest_time_utc": payload.get("ingest_time_utc"),
    })

endpoint_audit_df = pd.DataFrame(endpoint_rows)
endpoint_audit_df

,logical_name,source_doc_url,source_data_url,rows_from_json,pages,limit,ingest_time_utc
0,raw_garis_kemiskinan,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,594,1,5000,2026-06-05T15:50:40.827657Z
1,raw_persentase_miskin,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,405,1,5000,2026-06-05T15:50:41.560361Z
2,raw_keparahan_kemiskinan,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,567,1,5000,2026-06-05T15:50:42.262887Z
3,raw_ipm_sp2010,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,405,1,5000,2026-06-05T15:50:42.985793Z
4,raw_pengangguran_terbuka,https://data.jabarprov.go.id/api-backend/stati...,https://data.jabarprov.go.id/api-backend/bigda...,424,1,5000,2026-06-05T15:50:43.718883Z


#### 4. Memanggil Script Extraction

Sel ini adalah titik panggil ke `scripts/extract_api.py`. Untuk mode presentasi, `RUN_EXTRACT_PIPELINE` dibuat `False` agar notebook tidak otomatis melakukan request ulang ke API setiap kali dijalankan. Jika ingin memperbarui data dari Open Data Jabar, ubah nilainya menjadi `True`, lalu jalankan sel ini.


In [4]:
RUN_EXTRACT_PIPELINE = False

command = [sys.executable, str(EXTRACT_SCRIPT)]
print("Command extraction:", " ".join(command))

if RUN_EXTRACT_PIPELINE:
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Extraction gagal dengan return code {result.returncode}")
else:
    print("Mode presentasi: extraction tidak dijalankan ulang. Notebook memakai raw data latest yang sudah tersedia.")

Command extraction: /opt/miniconda3/bin/python /Users/refinormdn/data-analytics-tubes/scripts/extract_api.py
Mode presentasi: extraction tidak dijalankan ulang. Notebook memakai raw data latest yang sudah tersedia.


#### 5. Audit Output Raw Data

Audit ini mengecek apakah script extraction sudah menghasilkan pasangan file JSON dan CSV untuk setiap dataset, baik versi latest maupun versi timestamp. File timestamp berguna untuk tracking perubahan data dari waktu ke waktu, sedangkan file latest memudahkan tahap preprocessing membaca data terbaru.


In [5]:
raw_audit_rows = []
for logical_name in extract_module.api_docs:
    latest_csv = RAW_DIR / f"{logical_name}.csv"
    latest_json = RAW_DIR / f"{logical_name}.json"
    versioned_csv = sorted(RAW_DIR.glob(f"{logical_name}_*.csv"))
    versioned_json = sorted(RAW_DIR.glob(f"{logical_name}_*.json"))
    summary = summarize_latest_csv(logical_name)
    raw_audit_rows.append({
        "logical_name": logical_name,
        "latest_csv_exists": latest_csv.exists(),
        "latest_json_exists": latest_json.exists(),
        "versioned_csv_count": len(versioned_csv),
        "versioned_json_count": len(versioned_json),
        "rows": summary["rows"],
        "columns": summary["columns"],
        "tahun_min": summary["tahun_min"],
        "tahun_max": summary["tahun_max"],
    })

raw_audit_df = pd.DataFrame(raw_audit_rows)
raw_audit_df

,logical_name,latest_csv_exists,latest_json_exists,versioned_csv_count,versioned_json_count,rows,columns,tahun_min,tahun_max
0,raw_garis_kemiskinan,True,True,2,2,594,8,2004,2025
1,raw_persentase_miskin,True,True,2,2,405,8,2010,2024
2,raw_keparahan_kemiskinan,True,True,2,2,567,8,2004,2024
3,raw_ipm_sp2010,True,True,2,2,405,8,2010,2024
4,raw_pengangguran_terbuka,True,True,2,2,424,8,2007,2024


#### 6. Manifest, Checksum, dan Idempotensi

Manifest mencatat riwayat ingest. Kolom `checksum` dipakai untuk mendeteksi apakah data yang baru diambil sama dengan data yang sudah pernah disimpan. Jika checksum sama, script menandai proses sebagai `skipped`, sehingga pipeline bisa dijalankan ulang tanpa menumpuk file duplikat yang isinya sama.


In [6]:
manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df["ingest_time"] = pd.to_datetime(manifest_df["ingest_time"], errors="coerce")
manifest_latest_df = (
    manifest_df.sort_values("ingest_time")
    .groupby("logical_name", as_index=False)
    .tail(1)
    .sort_values("logical_name")
)

status_summary_df = (
    manifest_df.groupby(["logical_name", "status"])
    .size()
    .reset_index(name="jumlah_event")
    .sort_values(["logical_name", "status"])
)

display(status_summary_df)
manifest_latest_df[["ingest_time", "logical_name", "stored_filename", "source_url", "status", "notes"]]

,logical_name,status,jumlah_event
0,raw_garis_kemiskinan,failed,1
1,raw_garis_kemiskinan,skipped,2
2,raw_garis_kemiskinan,success,2
3,raw_ipm_sp2010,failed,1
4,raw_ipm_sp2010,skipped,2
5,raw_ipm_sp2010,success,2
6,raw_keparahan_kemiskinan,failed,1
7,raw_keparahan_kemiskinan,skipped,2
8,raw_keparahan_kemiskinan,success,2
9,raw_pengangguran_terbuka,failed,1


,ingest_time,logical_name,stored_filename,source_url,status,notes
20,2026-06-05 15:50:40.873675,raw_garis_kemiskinan,raw_garis_kemiskinan_20260605T155040Z.json,https://data.jabarprov.go.id/api-backend/bigda...,success,rows=594;doc_url=https://data.jabarprov.go.id/...
23,2026-06-05 15:50:43.053958,raw_ipm_sp2010,raw_ipm_sp2010_20260605T155042Z.json,https://data.jabarprov.go.id/api-backend/bigda...,success,rows=405;doc_url=https://data.jabarprov.go.id/...
22,2026-06-05 15:50:42.281830,raw_keparahan_kemiskinan,raw_keparahan_kemiskinan_20260605T155042Z.json,https://data.jabarprov.go.id/api-backend/bigda...,success,rows=567;doc_url=https://data.jabarprov.go.id/...
24,2026-06-05 15:50:43.736885,raw_pengangguran_terbuka,raw_pengangguran_terbuka_20260605T155043Z.json,https://data.jabarprov.go.id/api-backend/bigda...,success,rows=424;doc_url=https://data.jabarprov.go.id/...
21,2026-06-05 15:50:41.575622,raw_persentase_miskin,raw_persentase_miskin_20260605T155041Z.json,https://data.jabarprov.go.id/api-backend/bigda...,success,rows=405;doc_url=https://data.jabarprov.go.id/...


#### 7. Preview Struktur Raw Data

Preview ini bukan tahap cleaning. Tujuannya hanya menunjukkan bahwa output Obtain sudah berupa tabel mentah yang bisa diteruskan ke tahap Scrub. Cleaning, standardisasi, integrasi, dan feature engineering dilakukan pada bagian Scrub di bawah.


In [7]:
for logical_name in extract_module.api_docs:
    csv_path = RAW_DIR / f"{logical_name}.csv"
    print(f"\n=== {logical_name} ===")
    if csv_path.exists():
        display(pd.read_csv(csv_path).head(3))
    else:
        print("File latest belum tersedia. Jalankan extraction terlebih dahulu.")


=== raw_garis_kemiskinan ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,garis_kemiskinan,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,130927,RUPIAH/KAPITA/BULAN,2004
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,111202,RUPIAH/KAPITA/BULAN,2004
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,121902,RUPIAH/KAPITA/BULAN,2004



=== raw_persentase_miskin ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,persentase_penduduk_miskin,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,9.97,PERSEN,2010
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,10.65,PERSEN,2010
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,14.32,PERSEN,2010



=== raw_keparahan_kemiskinan ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,indeks_keparahan_kemiskinan,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,0.62,POIN,2004
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,0.68,POIN,2004
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,0.52,POIN,2004



=== raw_ipm_sp2010 ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,indeks_pembangunan_manusia,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,64.35,POIN,2010
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,60.69,POIN,2010
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,58.58,POIN,2010



=== raw_pengangguran_terbuka ===


,id,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tingkat_pengangguran_terbuka,satuan,tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,14.26,PERSEN,2007
1,2,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,10.85,PERSEN,2007
2,3,32,JAWA BARAT,3203,KABUPATEN CIANJUR,13.82,PERSEN,2007


#### 8. Proof Orkestrasi

Repository juga menyediakan scaffold Prefect pada `scripts/orchestrate_prefect.py`. Notebook ini tidak menjalankan Prefect secara langsung karena package `prefect` bersifat opsional, tetapi sel berikut mengecek bahwa orchestrator memang mendefinisikan task, flow, dan memanggil script extraction yang sama.


In [8]:
orchestrator_text = ORCHESTRATE_SCRIPT.read_text(encoding="utf-8")
orchestration_checks_df = pd.DataFrame([
    {"check": "File orchestrator tersedia", "status": ORCHESTRATE_SCRIPT.exists()},
    {"check": "Mendefinisikan Prefect task", "status": "@task" in orchestrator_text},
    {"check": "Mendefinisikan Prefect flow", "status": "@flow" in orchestrator_text},
    {"check": "Memanggil scripts/extract_api.py", "status": "scripts/extract_api.py" in orchestrator_text},
    {"check": "Memakai interpreter Python aktif", "status": "sys.executable" in orchestrator_text},
])

orchestration_checks_df

,check,status
0,File orchestrator tersedia,True
1,Mendefinisikan Prefect task,True
2,Mendefinisikan Prefect flow,True
3,Memanggil scripts/extract_api.py,True
4,Memakai interpreter Python aktif,True


#### 9. Ringkasan Obtain

Tahap Obtain sudah menyediakan data mentah dari API Open Data Jabar untuk lima indikator utama. Output disimpan sebagai JSON/CSV raw, dilengkapi metadata endpoint, manifest ingest, checksum, dan scaffold orkestrasi. Dengan struktur ini, tahap Scrub dapat memakai file latest dari `data/raw/`, sementara tim tetap bisa menjalankan ulang extraction jika sumber API diperbarui.


# **2. Scrub**

## *Data Preprocessing Lead* / 18223070

#### 1. Setup

In [9]:
from pathlib import Path
import json
import sys

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "scripts").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from scripts import preprocess_data as pp

#### 2. Dataset Yang Diproses

Dataset berikut dipakai untuk membangun panel kemiskinan dan kerentanan sosial per kabupaten/kota dan tahun.

In [10]:
dataset_catalog = pd.DataFrame([
    {
        "logical_name": spec.logical_name,
        "metric_col": spec.metric_col,
        "metric_label": spec.metric_label,
        "unit_hint": spec.unit_hint,
        "risk_direction": spec.risk_direction,
    }
    for spec in pp.DATASETS
])

dataset_catalog

,logical_name,metric_col,metric_label,unit_hint,risk_direction
0,raw_garis_kemiskinan,garis_kemiskinan,Garis kemiskinan,rupiah/kapita/bulan,context_only
1,raw_persentase_miskin,persentase_penduduk_miskin,Persentase penduduk miskin,persen,high_is_risky
2,raw_keparahan_kemiskinan,indeks_keparahan_kemiskinan,Indeks keparahan kemiskinan,indeks,high_is_risky
3,raw_ipm_sp2010,indeks_pembangunan_manusia,Indeks pembangunan manusia,indeks,low_is_risky
4,raw_pengangguran_terbuka,tingkat_pengangguran_terbuka,Tingkat pengangguran terbuka,persen,high_is_risky


#### 3. Profiling Raw Data

Bagian ini adalah Explore ringan untuk preprocessing. Tujuannya bukan mencari insight final, tapi memastikan sumber data bisa dibersihkan dan diintegrasikan.

In [11]:
def latest_matching_file(logical_name, suffix):
    candidates = []
    for directory in [pp.DEFAULT_SOURCE_DIR, pp.DEFAULT_ARCHIVE_DIR]:
        candidates.extend(directory.glob(f"{logical_name}{suffix}"))
        candidates.extend(directory.glob(f"{logical_name}_*{suffix}"))
    if not candidates:
        return None
    return sorted(candidates, key=lambda path: (path.stat().st_mtime, path.name), reverse=True)[0]


raw_profiles = []

for spec in pp.DATASETS:
    csv_path = latest_matching_file(spec.logical_name, ".csv")
    json_path = latest_matching_file(spec.logical_name, ".json")

    csv_rows = None
    csv_cols = []
    csv_status = "not_found"
    if csv_path is not None:
        try:
            csv_df = pd.read_csv(csv_path)
            csv_rows = len(csv_df)
            csv_cols = list(csv_df.columns)
            csv_status = "usable_columns" if pp.source_is_usable(csv_df, spec) else "not_statistical_records"
        except Exception as exc:
            csv_status = f"read_error: {exc}"

    json_kind = "not_found"
    json_records = None
    if json_path is not None:
        try:
            with json_path.open("r", encoding="utf-8") as handle:
                payload = json.load(handle)
            if isinstance(payload, dict) and "openapi" in payload:
                json_kind = "openapi_doc"
            else:
                records = pp.extract_records_from_json(payload)
                json_kind = "statistical_records" if records else "unknown_json_shape"
                json_records = len(records)
        except Exception as exc:
            json_kind = f"read_error: {exc}"

    raw_profiles.append({
        "logical_name": spec.logical_name,
        "expected_metric": spec.metric_col,
        "csv_file": str(csv_path.relative_to(PROJECT_ROOT)) if csv_path else "",
        "csv_rows": csv_rows,
        "csv_columns": ", ".join(csv_cols[:8]),
        "csv_status": csv_status,
        "json_file": str(json_path.relative_to(PROJECT_ROOT)) if json_path else "",
        "json_kind": json_kind,
        "json_records": json_records,
    })

raw_profile_df = pd.DataFrame(raw_profiles)
raw_profile_df

,logical_name,expected_metric,csv_file,csv_rows,csv_columns,csv_status,json_file,json_kind,json_records
0,raw_garis_kemiskinan,garis_kemiskinan,data/raw/raw_garis_kemiskinan_20260605T155040Z...,594,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/raw/raw_garis_kemiskinan_20260605T155040Z...,statistical_records,594
1,raw_persentase_miskin,persentase_penduduk_miskin,data/raw/raw_persentase_miskin_20260605T155041...,405,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/raw/raw_persentase_miskin_20260605T155041...,statistical_records,405
2,raw_keparahan_kemiskinan,indeks_keparahan_kemiskinan,data/raw/raw_keparahan_kemiskinan_20260605T155...,567,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/raw/raw_keparahan_kemiskinan_20260605T155...,statistical_records,567
3,raw_ipm_sp2010,indeks_pembangunan_manusia,data/raw/raw_ipm_sp2010_20260605T155042Z.csv,405,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/raw/raw_ipm_sp2010_20260605T155042Z.json,statistical_records,405
4,raw_pengangguran_terbuka,tingkat_pengangguran_terbuka,data/raw/raw_pengangguran_terbuka_20260605T155...,424,"id, kode_provinsi, nama_provinsi, kode_kabupat...",usable_columns,data/raw/raw_pengangguran_terbuka_20260605T155...,statistical_records,424


#### 4. Expected Schema Check

Untuk integrasi panel, setiap dataset minimal perlu punya key `kode_kabupaten_kota` dan `tahun`, nama wilayah, serta satu kolom metrik utama.

In [12]:
schema_checks = []

for spec in pp.DATASETS:
    csv_path = latest_matching_file(spec.logical_name, ".csv")
    expected_cols = pp.KEY_COLS + ["nama_kabupaten_kota", spec.metric_col]

    if csv_path is None:
        schema_checks.append({
            "logical_name": spec.logical_name,
            "expected_columns": ", ".join(expected_cols),
            "missing_columns": "all",
            "ready_for_scrub": False,
        })
        continue

    raw_df = pd.read_csv(csv_path)
    normalized_cols = set(pp.standardize_columns(raw_df).columns)
    missing_cols = [col for col in expected_cols if col not in normalized_cols]
    schema_checks.append({
        "logical_name": spec.logical_name,
        "expected_columns": ", ".join(expected_cols),
        "missing_columns": ", ".join(missing_cols),
        "ready_for_scrub": len(missing_cols) == 0,
    })

schema_check_df = pd.DataFrame(schema_checks)
schema_check_df

,logical_name,expected_columns,missing_columns,ready_for_scrub
0,raw_garis_kemiskinan,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
1,raw_persentase_miskin,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
2,raw_keparahan_kemiskinan,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
3,raw_ipm_sp2010,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True
4,raw_pengangguran_terbuka,"kode_kabupaten_kota, tahun, nama_kabupaten_kot...",,True


Jika `ready_for_scrub` bernilai `False` dan `json_kind` adalah `openapi_doc`, berarti file raw lokal masih berisi dokumentasi API, bukan record statistik. Notebook ini tetap bisa lanjut kalau internet tersedia dengan mengubah flag `FETCH_OPENAPI_IF_NEEDED` menjadi `True`.

#### 5. Cleaning Per Dataset

Keputusan preprocessing yang digunakan:
- nama kolom distandardisasi ke snake_case
- nama provinsi dan kabupaten/kota distandardisasi ke uppercase
- key `kode_kabupaten_kota` dan `tahun` dipaksa menjadi numerik integer
- metrik dipaksa menjadi numerik
- nilai 0 pada indikator inti diperlakukan sebagai placeholder missing jika tidak masuk akal secara domain
- duplikasi key kabupaten/kota-tahun dikurangi menjadi satu record
- missing value metrik hanya diimputasi untuk gap internal dengan interpolasi per wilayah; missing struktural tetap dibiarkan kosong

In [13]:
FETCH_OPENAPI_IF_NEEDED = False

cleaned_by_metric = {}
quality_profiles = []
source_infos = []
load_errors = []

for spec in pp.DATASETS:
    try:
        raw_df, source_info = pp.load_source_dataframe(
            spec=spec,
            source_dir=pp.DEFAULT_SOURCE_DIR,
            archive_dir=pp.DEFAULT_ARCHIVE_DIR,
            interim_dir=pp.DEFAULT_INTERIM_DIR,
            fetch_openapi_if_needed=FETCH_OPENAPI_IF_NEEDED,
            fetch_limit=5000,
            fetch_timeout=30,
        )
        cleaned_df, profile = pp.clean_dataset(raw_df, spec)
        cleaned_by_metric[spec.metric_col] = cleaned_df
        quality_profiles.append(profile)
        source_infos.append(source_info)
    except Exception as exc:
        load_errors.append({
            "logical_name": spec.logical_name,
            "metric": spec.metric_col,
            "error": str(exc),
        })

quality_profile_df = pd.DataFrame(quality_profiles)
load_error_df = pd.DataFrame(load_errors)

display(quality_profile_df)
display(load_error_df)

,logical_name,metric,rows_raw,rows_clean,missing_values_before,missing_values_after,missing_metric_before_impute,imputed_values,invalid_zero_values,rows_dropped_missing_keys,duplicate_key_rows,exact_duplicate_rows
0,raw_garis_kemiskinan,garis_kemiskinan,594,594,0,14,14,0,14,0,0,0
1,raw_persentase_miskin,persentase_penduduk_miskin,405,405,0,5,5,0,5,0,0,0
2,raw_keparahan_kemiskinan,indeks_keparahan_kemiskinan,567,567,0,14,14,0,14,0,0,0
3,raw_ipm_sp2010,indeks_pembangunan_manusia,405,405,0,3,3,0,3,0,0,0
4,raw_pengangguran_terbuka,tingkat_pengangguran_terbuka,424,424,0,0,0,0,0,0,0,0


""


#### 6. Integrasi Dataset dan Filter Periode

Semua dataset digabung menjadi satu panel dengan key `kode_kabupaten_kota` dan `tahun`, lalu difilter ke periode analisis `2010-2024` sesuai scope obtain.

In [14]:
expected_metric_count = len(pp.DATASETS)

ANALYSIS_START_YEAR = 2010
ANALYSIS_END_YEAR = 2024

if len(cleaned_by_metric) == expected_metric_count:
    panel = pp.integrate_datasets(cleaned_by_metric)
    panel = pp.filter_analysis_period(panel, ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
    pp.validate_panel(panel)
    print(f"Panel rows: {len(panel):,}")
    print(f"Kabupaten/kota: {panel['kode_kabupaten_kota'].nunique():,}")
    print(f"Year range: {int(panel['tahun'].min())} - {int(panel['tahun'].max())}")
    display(panel.head())
else:
    panel = None
    print("Panel belum bisa dibuat karena belum semua dataset berhasil dibaca dan dibersihkan.")

Panel rows: 405
Kabupaten/kota: 27
Year range: 2010 - 2024


,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,garis_kemiskinan,garis_kemiskinan_was_imputed,persentase_penduduk_miskin,persentase_penduduk_miskin_was_imputed,indeks_keparahan_kemiskinan,indeks_keparahan_kemiskinan_was_imputed,indeks_pembangunan_manusia,indeks_pembangunan_manusia_was_imputed,tingkat_pengangguran_terbuka,tingkat_pengangguran_terbuka_was_imputed
0,32,JAWA BARAT,3201,KABUPATEN BOGOR,2010,214338.0,False,9.97,False,0.43,False,64.35,False,10.64,False
1,32,JAWA BARAT,3202,KABUPATEN SUKABUMI,2010,184127.0,False,10.65,False,0.30,False,60.69,False,9.89,False
2,32,JAWA BARAT,3203,KABUPATEN CIANJUR,2010,202438.0,False,14.32,False,0.38,False,58.58,False,11.21,False
3,32,JAWA BARAT,3204,KABUPATEN BANDUNG,2010,217452.0,False,9.29,False,0.24,False,67.28,False,10.69,False
4,32,JAWA BARAT,3205,KABUPATEN GARUT,2010,180406.0,False,13.94,False,0.34,False,60.23,False,7.75,False


#### 7. Feature Engineering

Fitur yang dibuat:
- perubahan year-over-year tiap metrik
- persentase kemiskinan rolling 3 tahun
- arah tren kemiskinan
- skor kerentanan sosial berbasis z-score tahunan
- peringkat dan bucket prioritas intervensi
- jumlah indikator yang diimputasi

In [15]:
if panel is not None:
    panel_featured = pp.add_features(panel)
    pp.validate_panel(panel_featured)
    display(panel_featured.head(10))
else:
    panel_featured = None
    print("Feature engineering belum dijalankan karena panel belum tersedia.")

,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,garis_kemiskinan,garis_kemiskinan_was_imputed,persentase_penduduk_miskin,persentase_penduduk_miskin_was_imputed,indeks_keparahan_kemiskinan,indeks_keparahan_kemiskinan_was_imputed,indeks_pembangunan_manusia,indeks_pembangunan_manusia_was_imputed,tingkat_pengangguran_terbuka,tingkat_pengangguran_terbuka_was_imputed,garis_kemiskinan_yoy_change,garis_kemiskinan_yoy_pct_change,persentase_penduduk_miskin_yoy_change,persentase_penduduk_miskin_yoy_pct_change,indeks_keparahan_kemiskinan_yoy_change,indeks_keparahan_kemiskinan_yoy_pct_change,indeks_pembangunan_manusia_yoy_change,indeks_pembangunan_manusia_yoy_pct_change,tingkat_pengangguran_terbuka_yoy_change,tingkat_pengangguran_terbuka_yoy_pct_change,persentase_miskin_rolling_3y,arah_tren_kemiskinan,persentase_penduduk_miskin_risk_z,indeks_keparahan_kemiskinan_risk_z,indeks_pembangunan_manusia_risk_z,tingkat_pengangguran_terbuka_risk_z,jumlah_komponen_skor_kerentanan,skor_kerentanan_sosial,peringkat_prioritas_intervensi,prioritas_intervensi,jumlah_indikator_diimputasi,jumlah_indikator_tersedia
0,32,JAWA BARAT,3278,KOTA TASIKMALAYA,2010,263177.0,False,20.71,False,1.17,False,66.58,False,8.16,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.71,belum_ada_pembanding,2.352869,3.504758,-0.057965,-0.723118,4,1.2691,1,sangat_tinggi,0,5
1,32,JAWA BARAT,3215,KABUPATEN KARAWANG,2010,266597.0,False,12.21,False,0.78,False,64.58,False,14.88,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.21,belum_ada_pembanding,0.211695,1.595126,0.324059,1.492651,4,0.9059,2,sangat_tinggi,0,5
2,32,JAWA BARAT,3217,KABUPATEN BANDUNG BARAT,2010,216388.0,False,14.68,False,0.58,False,61.34,False,13.31,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.68,belum_ada_pembanding,0.833895,0.615828,0.942938,0.974979,4,0.8419,3,sangat_tinggi,0,5
3,32,JAWA BARAT,3209,KABUPATEN CIREBON,2010,230346.0,False,16.12,False,0.58,False,63.64,False,12.97,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.12,belum_ada_pembanding,1.196635,0.615828,0.503611,0.862871,4,0.7947,4,sangat_tinggi,0,5
4,32,JAWA BARAT,3212,KABUPATEN INDRAMAYU,2010,264576.0,False,16.58,False,0.52,False,60.86,False,11.29,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.58,belum_ada_pembanding,1.312510,0.322038,1.034624,0.308929,4,0.7445,5,sangat_tinggi,0,5
5,32,JAWA BARAT,3203,KABUPATEN CIANJUR,2010,202438.0,False,14.32,False,0.38,False,58.58,False,11.21,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.32,belum_ada_pembanding,0.743210,-0.363470,1.470132,0.282551,4,0.5331,6,tinggi,0,5
6,32,JAWA BARAT,3210,KABUPATEN MAJALENGKA,2010,263377.0,False,15.51,False,0.67,False,62.30,False,5.82,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.51,belum_ada_pembanding,1.042974,1.056512,0.759567,-1.494680,4,0.3411,7,tinggi,0,5
7,32,JAWA BARAT,3272,KOTA SUKABUMI,2010,284339.0,False,9.24,False,0.52,False,67.94,False,15.65,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.24,belum_ada_pembanding,-0.536456,0.322038,-0.317741,1.746541,4,0.3036,8,tinggi,0,5
8,32,JAWA BARAT,3271,KOTA BOGOR,2010,278530.0,False,9.47,False,0.48,False,71.25,False,17.20,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.47,belum_ada_pembanding,-0.478518,0.126179,-0.949991,2.257619,4,0.2388,9,tinggi,0,5
9,32,JAWA BARAT,3213,KABUPATEN SUBANG,2010,234803.0,False,13.54,False,0.52,False,63.54,False,8.72,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.54,belum_ada_pembanding,0.546726,0.322038,0.522712,-0.538470,4,0.2133,10,tinggi,0,5


#### 8. Validasi Output Preprocessing

Validasi ini memastikan output tidak punya duplikasi key dan menunjukkan missing value yang masih tersisa setelah preprocessing.

In [16]:
if panel_featured is not None:
    validation_summary = {
        "rows": len(panel_featured),
        "duplicate_kabupaten_kota_year_keys": int(panel_featured.duplicated(pp.KEY_COLS).sum()),
        "missing_key_values": int(panel_featured[pp.KEY_COLS].isna().sum().sum()),
        "kabupaten_kota_count": int(panel_featured["kode_kabupaten_kota"].nunique()),
        "year_min": int(panel_featured["tahun"].min()),
        "year_max": int(panel_featured["tahun"].max()),
    }
    display(pd.DataFrame([validation_summary]))
    display(panel_featured.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))
else:
    print("Validasi output belum bisa dijalankan karena panel final belum tersedia.")

,rows,duplicate_kabupaten_kota_year_keys,missing_key_values,kabupaten_kota_count,year_min,year_max
0,405,0,0,27,2010,2024


,missing_count
tingkat_pengangguran_terbuka_yoy_pct_change,86
tingkat_pengangguran_terbuka_yoy_change,86
persentase_penduduk_miskin_yoy_pct_change,32
tingkat_pengangguran_terbuka_risk_z,32
indeks_keparahan_kemiskinan_yoy_pct_change,32
indeks_keparahan_kemiskinan_yoy_change,32
persentase_penduduk_miskin_yoy_change,32
garis_kemiskinan_yoy_pct_change,32
garis_kemiskinan_yoy_change,32
tingkat_pengangguran_terbuka_was_imputed,32


#### 9. Preview Prioritas Intervensi

Preview ini bukan pengganti analisis akhir, tapi membantu memastikan fitur prioritas sudah terbentuk dan masuk akal secara struktur.

In [17]:
if panel_featured is not None:
    scored_years = panel_featured.loc[panel_featured["skor_kerentanan_sosial"].notna(), "tahun"]
    latest_year = scored_years.max() if not scored_years.empty else panel_featured["tahun"].max()
    preview_cols = [
        "tahun",
        "peringkat_prioritas_intervensi",
        "nama_kabupaten_kota",
        "skor_kerentanan_sosial",
        "prioritas_intervensi",
        "persentase_penduduk_miskin",
        "indeks_keparahan_kemiskinan",
        "tingkat_pengangguran_terbuka",
        "indeks_pembangunan_manusia",
    ]
    display(
        panel_featured.loc[
            panel_featured["tahun"].eq(latest_year) & panel_featured["skor_kerentanan_sosial"].notna(),
            preview_cols,
        ]
        .sort_values("peringkat_prioritas_intervensi")
        .head(10)
    )
else:
    print("Preview prioritas belum tersedia karena panel final belum tersedia.")

,tahun,peringkat_prioritas_intervensi,nama_kabupaten_kota,skor_kerentanan_sosial,prioritas_intervensi,persentase_penduduk_miskin,indeks_keparahan_kemiskinan,tingkat_pengangguran_terbuka,indeks_pembangunan_manusia
378,2024,1,KABUPATEN KUNINGAN,1.2018,sangat_tinggi,11.88,0.53,7.78,71.26
379,2024,2,KABUPATEN INDRAMAYU,1.0761,sangat_tinggi,11.93,0.54,6.25,69.83
380,2024,3,KABUPATEN CIANJUR,0.7443,sangat_tinggi,10.14,0.41,5.99,67.24
381,2024,4,KABUPATEN SUBANG,0.6687,sangat_tinggi,9.49,0.46,6.73,71.36
382,2024,5,KABUPATEN CIREBON,0.6097,sangat_tinggi,11.00,0.36,6.74,71.44
383,2024,6,KABUPATEN GARUT,0.5188,tinggi,9.68,0.29,6.96,68.79
384,2024,7,KABUPATEN MAJALENGKA,0.4586,tinggi,10.82,0.45,4.01,69.74
385,2024,8,KABUPATEN SUMEDANG,0.3969,tinggi,9.10,0.45,6.16,73.73
386,2024,9,KABUPATEN BANDUNG BARAT,0.3696,tinggi,10.49,0.23,6.70,70.03
387,2024,10,KABUPATEN PURWAKARTA,0.3641,tinggi,8.41,0.35,7.34,72.65


#### 10. Simpan Output

Default cell ini dibuat **read-only** untuk mode presentasi notebook. Output preprocessing hanya akan ditulis ulang ke `data/processed` dan `reports/preprocessing` jika `WRITE_OUTPUTS` diubah menjadi `True`.


In [18]:
WRITE_OUTPUTS = False

if WRITE_OUTPUTS and panel_featured is not None:
    pp.DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pp.DEFAULT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

    panel_path = pp.DEFAULT_OUTPUT_DIR / "panel_kemiskinan_jabar_preprocessed.csv"
    dictionary_path = pp.DEFAULT_OUTPUT_DIR / "feature_dictionary.csv"

    panel_featured.to_csv(panel_path, index=False)
    pd.DataFrame(pp.feature_dictionary_rows()).to_csv(dictionary_path, index=False)
    manifest_path = pp.append_manifest(pp.DEFAULT_OUTPUT_DIR, panel_path, source_infos, panel_featured)
    summary_path, report_path = pp.write_quality_outputs(
        quality_profiles,
        source_infos,
        panel_featured,
        pp.DEFAULT_OUTPUT_DIR,
        pp.DEFAULT_REPORT_DIR,
    )

    print("Output tersimpan:")
    for path in [panel_path, dictionary_path, manifest_path, summary_path, report_path]:
        print(f"- {path.relative_to(PROJECT_ROOT)}")
elif panel_featured is None:
    print("Output belum disimpan karena panel final belum tersedia.")
else:
    print("WRITE_OUTPUTS = False, output tidak disimpan.")

WRITE_OUTPUTS = False, output tidak disimpan.


# 3. *Explore*

# 4. *Model*

# 5. *iNterpret*